# Assignment 1

You only need to write one line of code for each question. When answering questions that ask you to identify or interpret something, the length of your response doesn’t matter. For example, if the answer is just ‘yes,’ ‘no,’ or a number, you can just give that answer without adding anything else.

We will go through comparable code and concepts in the live learning session. If you run into trouble, start by using the help `help()` function in Python, to get information about the datasets and function in question. The internet is also a great resource when coding (though note that **no outside searches are required by the assignment!**). If you do incorporate code from the internet, please cite the source within your code (providing a URL is sufficient).

Please bring questions that you cannot work out on your own to office hours, work periods or share with your peers on Slack. We will work with you through the issue.

### Classification using KNN

Let's set up our workspace and use the **Wine dataset** from `scikit-learn`. This dataset contains 178 wine samples with 13 chemical features, used to classify wines into different classes based on their origin.

The **response variable** is `class`, which indicates the type of wine. We'll use all of the chemical features to predict this response variable.

In [13]:
# Import standard libraries
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import recall_score, precision_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

In [14]:
from sklearn.datasets import load_wine

# Load the Wine dataset
wine_data = load_wine()

# Convert to DataFrame
wine_df = pd.DataFrame(wine_data.data, columns=wine_data.feature_names)

# Bind the 'class' (wine target) to the DataFrame
wine_df['class'] = wine_data.target

# Display the DataFrame
wine_df


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,class
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,13.71,5.65,2.45,20.5,95.0,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740.0,2
174,13.40,3.91,2.48,23.0,102.0,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750.0,2
175,13.27,4.28,2.26,20.0,120.0,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835.0,2
176,13.17,2.59,2.37,20.0,120.0,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840.0,2


#### **Question 1:** 
#### Data inspection

Before fitting any model, it is essential to understand our data. **Use Python code** to answer the following questions about the **Wine dataset**:

_(i)_ How many observations (rows) does the dataset contain?

In [15]:
# Your answer here

num_rows = wine_df.shape[0]

print(f"The Wine dataset contains {num_rows} rows.")

The Wine dataset contains 178 rows.


In [16]:
# My answer 2
print(f"The Wine dataset contains the number of rows: {wine_df.shape[0]}")

The Wine dataset contains the number of rows: 178


_(ii)_ How many variables (columns) does the dataset contain?

In [17]:
# Your answer here
num_columns = wine_df.shape[1]
print(f"The Wine dataset contains {num_columns} columns.")

The Wine dataset contains 14 columns.


In [18]:
# My answer 2
print(f"The Wine dataset contains {wine_df.shape[1]} columns.")

The Wine dataset contains 14 columns.


_(iii)_ What is the 'variable type' of the response variable `class` (e.g., 'integer', 'category', etc.)? What are the 'levels' (unique values) of the variable?

In [19]:
# Your answer here
# Extract the target variable (response variable) `class`
response_variable = wine_data.target

# Determine the variable type
variable_type = type(response_variable[0])

# Determine the levels (unique values) of the variable
levels = pd.unique(response_variable)

print(f"The variable type of the response variable `class` is {variable_type.__name__}.")
print(f"The levels (unique values) of the response variable `class` are: {list(levels)}.")

The variable type of the response variable `class` is int32.
The levels (unique values) of the response variable `class` are: [0, 1, 2].


In [20]:
# My answer 2:

print(f"The variable type of the response variable `class` is {type(response_variable[0]).__name__}.")
print(f"The levels (unique values) of the response variable `class` are: {list(pd.unique(response_variable))}.")

The variable type of the response variable `class` is int32.
The levels (unique values) of the response variable `class` are: [0, 1, 2].



_(iv)_ How many predictor variables do we have (Hint: all variables other than `class`)? 

In [21]:
# Your answer here
num_predictor_variables = wine_df.shape[1]

print(f"The Wine dataset contains {num_predictor_variables-1} predictor variables.")

The Wine dataset contains 13 predictor variables.


In [22]:
# or
print(f"The Wine dataset contains {wine_df.shape[1]-1} predictor variables.")

The Wine dataset contains 13 predictor variables.


You can use `print()` and `describe()` to help answer these questions.

#### **Question 2:** 
#### Standardization and data-splitting

Next, we must preform 'pre-processing' or 'data munging', to prepare our data for classification/prediction. For KNN, there are three essential steps. A first essential step is to 'standardize' the predictor variables. We can achieve this using the scaler method, provided as follows:

In [23]:
# Select predictors (excluding the last column)
predictors = wine_df.iloc[:, :-1]
#prediction = wine_df["class"]
# Standardize the predictors
scaler = StandardScaler()
predictors_standardized = pd.DataFrame(scaler.fit_transform(predictors), columns=predictors.columns)

# Display the head of the standardized predictors
print(predictors_standardized.head())

    alcohol  malic_acid       ash  alcalinity_of_ash  magnesium  \
0  1.518613   -0.562250  0.232053          -1.169593   1.913905   
1  0.246290   -0.499413 -0.827996          -2.490847   0.018145   
2  0.196879    0.021231  1.109334          -0.268738   0.088358   
3  1.691550   -0.346811  0.487926          -0.809251   0.930918   
4  0.295700    0.227694  1.840403           0.451946   1.281985   

   total_phenols  flavanoids  nonflavanoid_phenols  proanthocyanins  \
0       0.808997    1.034819             -0.659563         1.224884   
1       0.568648    0.733629             -0.820719        -0.544721   
2       0.808997    1.215533             -0.498407         2.135968   
3       2.491446    1.466525             -0.981875         1.032155   
4       0.808997    0.663351              0.226796         0.401404   

   color_intensity       hue  od280/od315_of_diluted_wines   proline  
0         0.251717  0.362177                      1.847920  1.013009  
1        -0.293321  0.406051

In [24]:
predictors_standardized.shape

(178, 13)

In [25]:
response_variable.shape

(178,)

(i) Why is it important to standardize the predictor variables?

> Your answer here...Standardizing predictor variables is important in statistical modeling and machine learning for several reasons:

1. Comparability of Variables
Standardization ensures that variables with different units and scales are made comparable. 
2. To ensure all predictor variables have fair contribution to the model.
Standardization prevents one variable from being overpower or disproportionately influencing the model simply because of its larger scale.


(i) Why is it important to standardize the predictor variables?

(ii) Why did we elect not to standard our response variable `Class`?

> Your answer here...Here are the reasons:
1. For the nature of the Response Variable:
The response variable Class is categorical. Standardization is not applicable or necessary for categorical variables because the values are labels rather than numerical quantities.
Standardization is not applicable or necessary for binary variables because the values are already bounded and carry their intended meaning without needing transformation.
2. For the purpose of Standardization
Standardization is typically applied to predictor variables to address issues of scale and comparability. The response variable does not need to be scaled for these purposes because it serves as the target that the model tries to predict, not a feature influencing the prediction.

(iii) A second essential step is to set a random seed. Do so below (Hint: use the random.seed function). Why is setting a seed important? Is the particular seed value important? Why or why not?

> Your answer here...
In Python, seed can be coded as:
import random
random.seed (123)

1. Why is Setting a Seed Important?
Setting a random seed is critical for reproducibility and consistency in experiments involving randomness.
1.1 Reproducibility:
Setting a random seed ensures that the sequence of random numbers or results of randomized operations (e.g., shuffling data, splitting datasets, initializing model weights) remains the same every time the code is run. This is crucial for replicating results in scientific research, debugging, or when sharing work with others.
1.2 Consistency Across Runs:
Random processes without a fixed seed produce different outcomes each time the code is executed. Setting a seed guarantees consistent results, which simplifies testing and validation.
1.3 Fair Comparisons: 
A fixed seed ensures that randomness does not bias the comparison.

2. Is the Particular Seed Value Important? Why or why not?
2.1 Not important for the value of the seed:
The specific seed value itself doesn’t matter as long as it is fixed. Any seed (e.g., 42, 123, 2023) will produce consistent results as long as the same value is used across runs.
2.2 Using same value for seed is important for reproucibility of generated sequences:
Changing the seed will produce a different sequence of random numbers, which might lead to slightly different results in some contexts (e.g., model performance, shuffling).It must be fixed and documented to ensure others can replicate your results accurately.


(iv) A third essential step is to split our standardized data into separate training and testing sets. We will split into 75% training and 25% testing. The provided code randomly partitions our data, and creates linked training sets for the predictors and response variables. 

Extend the code to create a non-overlapping test set for the predictors and response variables.

In [26]:
#  set a seed for reproducibility
np.random.seed(123)


# split the data into a training and testing set. hint: use train_test_split !
x_train, x_test= train_test_split(
    predictors_standardized, train_size=0.75, shuffle = True,
    random_state=123
)

y_train, y_test= train_test_split(
    response_variable, train_size=0.75, shuffle = True,
    random_state=123
)

#### **Question 3:**
#### Model initialization and cross-validation
We are finally set to fit the KNN model. 


Perform a grid search to tune the `n_neighbors` hyperparameter using 10-fold cross-validation. Follow these steps:

1. Initialize the KNN classifier using `KNeighborsClassifier()`.
2. Define a parameter grid for `n_neighbors` ranging from 1 to 50.
3. Implement a grid search using `GridSearchCV` with 10-fold cross-validation to find the optimal number of neighbors.
4. After fitting the model on the training data, identify and return the best value for `n_neighbors` based on the grid search results.

In [38]:
# Your code here..
# Create model
knn = KNeighborsClassifier(n_neighbors=3)
# cross validation
returned_dictionary = cross_validate(
    estimator= knn,
    cv= 10,
    X = X,
    y = y
)
cv_10_df = pd.DataFrame(returned_dictionary)
cv_10_df

# Compute mean and standard error of the mean (SEM) for each column
cv_10_metrics = cv_10_df.agg(["mean", "sem"])

cv_10_metrics

# The estimated accuracy, presented by Mean, whose test_score is 0.962; 
# The uncertanty around this estimate, presented by sem, whose test_core is 0.017
# So the true average accuracy is like between 94.5% and 97.9% 

,fit_time,score_time,test_score
mean,0.002455,0.005766,0.961538
sem,0.000566,0.001253,0.017201


In [34]:
# To find the best K
# create our paremeter grid
knn = KNeighborsClassifier()
parameter_grid = {
    "n_neighbors": range(1,50,5)
}
parameter_grid

wine_tune_grid = GridSearchCV(
    estimator = knn,
    param_grid = parameter_grid,
    cv = 10        
)

wine_tune_grid.fit(x_train,y_train)


GridSearchCV(cv=10, estimator=KNeighborsClassifier(),
             param_grid={'n_neighbors': range(1, 50, 5)})

In [35]:
accuracy_grid = pd.DataFrame(wine_tune_grid.cv_results_)
accuracy_grid

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_n_neighbors,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,split5_test_score,split6_test_score,split7_test_score,split8_test_score,split9_test_score,mean_test_score,std_test_score,rank_test_score
0,0.002602,0.001265,0.005404,0.002758,1,{'n_neighbors': 1},1.0,0.928571,0.928571,1.0,0.923077,1.000000,0.923077,1.0,0.769231,0.923077,0.939560,0.066844,10
1,0.002182,0.001483,0.003037,0.002169,6,{'n_neighbors': 6},1.0,1.000000,1.000000,1.0,0.923077,1.000000,1.000000,1.0,0.769231,1.000000,0.969231,0.070501,7
2,0.002348,0.001906,0.002437,0.001304,11,{'n_neighbors': 11},1.0,1.000000,1.000000,1.0,0.923077,1.000000,1.000000,1.0,0.846154,0.923077,0.969231,0.051025,7
3,0.001762,0.003361,0.002911,0.004150,16,{'n_neighbors': 16},1.0,1.000000,1.000000,1.0,0.923077,1.000000,1.000000,1.0,0.923077,0.923077,0.976923,0.035251,3
4,0.001135,0.000844,0.003694,0.002667,21,{'n_neighbors': 21},1.0,1.000000,1.000000,1.0,1.000000,1.000000,1.000000,1.0,0.923077,0.923077,0.984615,0.030769,1
5,0.002292,0.004006,0.002006,0.001128,26,{'n_neighbors': 26},1.0,1.000000,1.000000,1.0,1.000000,1.000000,1.000000,1.0,0.923077,0.923077,0.984615,0.030769,1
6,0.001441,0.001165,0.003353,0.002798,31,{'n_neighbors': 31},1.0,1.000000,1.000000,1.0,1.000000,1.000000,1.000000,1.0,0.846154,0.923077,0.976923,0.049255,3
7,0.002216,0.004413,0.002483,0.004380,36,{'n_neighbors': 36},1.0,0.928571,1.000000,1.0,1.000000,1.000000,1.000000,1.0,0.923077,0.846154,0.969780,0.050552,5
8,0.000799,0.000772,0.002885,0.003791,41,{'n_neighbors': 41},1.0,0.928571,1.000000,1.0,1.000000,0.923077,1.000000,1.0,0.923077,0.923077,0.969780,0.037042,5
9,0.002037,0.004478,0.002564,0.004671,46,{'n_neighbors': 46},1.0,0.857143,1.000000,1.0,0.923077,0.923077,1.000000,1.0,0.923077,0.846154,0.947253,0.058314,9


In [29]:
# Find the best n_neighbors
wine_tune_grid.best_params_ 
# Return {'n_neighbors': 21}

#My question: If set parameter_grid = { "n_neighbors": range(1,50,2)}, will return {'n_neighbors': 15}. Which would be the right one?

{'n_neighbors': 21}

#### **Question 4:**
#### Model evaluation

Using the best value for `n_neighbors`, fit a KNN model on the training data and evaluate its performance on the test set using `accuracy_score`.

In [30]:
# Your code here...
# Step 1. creating our model
knn = KNeighborsClassifier(n_neighbors=21)
X = x_train
y = y_train



In [ ]:
knn.fit(x_train,y_train)

In [ ]:
accuracy_score = knn.score(x_test,y_test)
accuracy_score
# the output shows the estimated accuracy of the classifier on the test data was 93.33%

0.9333333333333333

# Criteria


| **Criteria**                                           | **Complete**                                      | **Incomplete**                                    |
|--------------------------------------------------------|---------------------------------------------------|--------------------------------------------------|
| **Data Inspection**                                    | Data is inspected for number of variables, observations and data types. | Data inspection is missing or incomplete.         |
| **Data Scaling**                                       | Data scaling or normalization is applied where necessary (e.g., using `StandardScaler`). | Data scaling or normalization is missing or incorrectly applied. |
| **Model Initialization**                               | The KNN model is correctly initialized and a random seed is set for reproducibility.            | The KNN model is not initialized, is incorrect, or lacks a random seed for reproducibility. |
| **Parameter Grid for `n_neighbors`**                   | The parameter grid for `n_neighbors` is correctly defined. | The parameter grid is missing or incorrectly defined. |
| **Cross-Validation Setup**                             | Cross-validation is set up correctly with 10 folds. | Cross-validation is missing or incorrectly set up. |
| **Best Hyperparameter (`n_neighbors`) Selection**       | The best value for `n_neighbors` is identified using the grid search results. | The best `n_neighbors` is not selected or incorrect. |
| **Model Evaluation on Test Data**                      | The model is evaluated on the test data using accuracy. | The model evaluation is missing or uses the wrong metric. |


## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Note:

If you like, you may collaborate with others in the cohort. If you choose to do so, please indicate with whom you have worked with in your pull request by tagging their GitHub username. Separate submissions are required.

### Submission Parameters:
* Submission Due Date: `11:59 PM - 01/12/2025`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/LCR/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-4-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
